# vit5 feature extraction — PanNuke

Single-checkpoint, single-dataset version of `run_ablation_sweep.py`, stripped
down to just the `vit5_small`/`vit5_base` loading path and the PanNuke dataset.
No manifest CSV, no multi-model dispatch, no sweep bookkeeping — just:
load checkpoint → build transform → set up PanNuke splits → extract & save
features to `.h5`.

Uses your existing `vit5_dino_wrapper.load_vit5_teacher_for_eval` and
`feature_extraction.setup_pannuke_datasets` / `save_features_combined`
unchanged, so behavior matches what the sweep script would have produced for
a PanNuke + vit5 manifest row.

In [ ]:
import sys
import torch
from pathlib import Path
from types import SimpleNamespace
from torch.utils.data import DataLoader
from torchvision import transforms as T

# --- adjust these two paths to your repo layout ---
VIT5_WRAPPER_DIR = "/tia/phd-data/u5552013/ImmunoDINO_baselines_comparison/cell_representation_learning/dinov2/models"
FEATURE_EXTRACTION_DIR = "/path/to/dir/containing/feature_extraction.py"
# ----------------------------------------------------

sys.path.insert(0, VIT5_WRAPPER_DIR)
sys.path.insert(0, FEATURE_EXTRACTION_DIR)

from vit5_dino_wrapper import load_vit5_teacher_for_eval
from feature_extraction import setup_pannuke_datasets, save_features_combined

## Config

In [ ]:
# Model / checkpoint
MODEL_NAME = "vit5_small"     # or "vit5_base"
PATCH_SIZE = 4                # matches cfg.student.patch_size from training
CROP_SIZE = 64                # matches DataAugmentationHEMA's global_crops_size used in training
CHECKPOINT_PATH = (
    "/tia/phd-data/u5552013/ImmunoDINO_baselines_comparison/imunocto_eval/"
    "dinov2_p4_vit5_kd_uni_tcga_nontcga/dinov2_p4_vit5/eval/training_30999/teacher_checkpoint.pth"
)

# Dataset / output
SAVE_DIR = Path("/path/to/ablation_features")   # where train/val/test .h5 files go
RUN_NAME = "vit5_small_p4_pannuke"
BATCH_SIZE = 256
NUM_WORKERS = 4

# Load-quality gate -- same 5% threshold run_ablation_sweep.py uses. Set to
# False to extract anyway even if a lot of tensors were missing/unexpected.
ENFORCE_LOAD_QUALITY = True
MISMATCH_THRESHOLD = 0.05

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}  |  Using device: {device}")
if device.type == "cuda":
    print(f"  GPU name: {torch.cuda.get_device_name(device.index or 0)}")
else:
    print("[WARNING] Running on CPU -- feature extraction will be much slower.")

## Load the model

Same call `load_model_and_transform()` makes for `model_name in ("vit5_small", "vit5_base")`
in the sweep script, minus the stdout-capturing (printed directly here instead).

In [ ]:
feature_extractor = load_vit5_teacher_for_eval(
    CHECKPOINT_PATH,
    modelname=MODEL_NAME,
    patch_size=PATCH_SIZE,
    force_sdpa_backend=True,
)
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

transform = T.Compose([
    T.Resize((CROP_SIZE, CROP_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
print("Model loaded and moved to", device)

### (Optional) sanity-check the load report

`load_vit5_teacher_for_eval` prints `missing keys (N): [...]` / `unexpected keys (N): [...]`
directly (see the function's own print statements) rather than returning them, so this
just re-derives the counts from a freshly re-run, captured call -- skip this cell if you
already eyeballed the printed output above and it looked clean.

In [ ]:
import io
import re
import contextlib

def parse_load_report(printed_output: str):
    n_total = None
    m = re.search(r"->\s*(\d+)\s*tensors", printed_output)
    if m:
        n_total = int(m.group(1))
    n_missing = 0
    m = re.search(r"missing keys \((\d+)\)", printed_output)
    if m:
        n_missing = int(m.group(1))
    n_unexpected = 0
    m = re.search(r"unexpected keys \((\d+)\)", printed_output)
    if m:
        n_unexpected = int(m.group(1))
    return {"n_total": n_total, "n_missing": n_missing, "n_unexpected": n_unexpected}

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    _ = load_vit5_teacher_for_eval(CHECKPOINT_PATH, modelname=MODEL_NAME, patch_size=PATCH_SIZE)
report = parse_load_report(buf.getvalue())
print(report)

if ENFORCE_LOAD_QUALITY and report["n_total"]:
    n_bad = report["n_missing"] + report["n_unexpected"]
    if n_bad >= MISMATCH_THRESHOLD * report["n_total"]:
        raise RuntimeError(
            f"{n_bad}/{report['n_total']} tensors missing/unexpected -- checkpoint doesn't "
            f"match MODEL_NAME/PATCH_SIZE as configured. Set ENFORCE_LOAD_QUALITY = False "
            f"to extract anyway."
        )

## Set up PanNuke datasets and dataloaders

In [ ]:
# setup_pannuke_datasets(args, transform) reads whichever of these attributes
# it needs -- passing the full set (matching run_ablation_sweep.py's argparse
# surface) so it works regardless of which ones PanNuke's setup actually uses.
args = SimpleNamespace(
    data_dir=None,
    csv_path=None,
    seed=42,
    hest_sample_ids=None,
    hest_fold=0,
    nucls_fold=1,
    nucls_label_type="super",
    nucls_keep_classes=None,
)

train_dataset, val_dataset, test_dataset = setup_pannuke_datasets(args, transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True) if val_dataset is not None else None
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

class_names = getattr(train_dataset, "classes", None)
if class_names is None and hasattr(train_dataset, "dataset"):
    class_names = getattr(train_dataset.dataset, "classes", None)

print(f"train: {len(train_dataset)}  val: {len(val_dataset) if val_dataset else 0}  test: {len(test_dataset)}")
print("classes:", class_names)

## Extract and save features

In [ ]:
output_base = SAVE_DIR / "pannuke" / RUN_NAME
output_base.mkdir(parents=True, exist_ok=True)

save_features_combined(feature_extractor, train_loader, output_base / "train_data", "train", class_names)
if val_loader is not None:
    save_features_combined(feature_extractor, val_loader, output_base / "val_data", "val", class_names)
save_features_combined(feature_extractor, test_loader, output_base / "test_data", "test", class_names)

print(f"Done. Features under {output_base}/{{train,val,test}}_data/")

## Evaluate features: logistic regression + kNN, with visualizations

Adapted from `train_and_evaluate_logistic_regression()` / `perform_knn()` in
your evaluation script -- same metrics, same logistic-regression
hyperparameters (`C = M*C/100`, `solver="lbfgs"`), same kNN `n_neighbors`
values (1, 20), and the same plot set (confusion matrix, ROC curves,
per-class F1, top-k accuracy, precision/recall, per-class AUROC, mAP,
Cohen's kappa, prediction-confidence distribution, per-class metrics grid,
feature importance). Every `wandb.log(...)` call has been swapped for
`plt.show()` so it renders inline here instead -- no wandb run required.
The k-fold CV and multi-run comparison-table code (`--table`, `--kfold`)
is left out; that's for comparing many ablation runs at once, not a
single checkpoint on one dataset.

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, cohen_kappa_score, confusion_matrix, f1_score,
    log_loss, precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.preprocessing import label_binarize

try:
    import umap
    _UMAP_AVAILABLE = True
except ImportError:
    _UMAP_AVAILABLE = False
    print("umap-learn not installed -- the UMAP cell at the end will be skipped. "
          "`pip install umap-learn` to enable it.")

DEFAULT_PANNUKE_CLASSES = ["Connective", "Dead", "Epithelial", "Inflammatory", "Neoplastic"]

### Metric functions (unchanged from your script)

In [ ]:
def load_combined_features(file_path):
    """Load features from a single combined HDF5 file."""
    with h5py.File(file_path, "r") as hf:
        features = hf["features"][:]
        labels = hf["labels"][:]
        ids = hf["ids"][:] if "ids" in hf else None
        class_names = None
        if "class_names" in hf:
            class_names = [c.decode("utf-8") if isinstance(c, bytes) else c for c in hf["class_names"][:]]
    return features, labels, ids, class_names


def topk_accuracy(y_true, y_pred_proba, k):
    top_k_preds = np.argsort(y_pred_proba, axis=1)[:, -k:]
    correct = sum(y_true[i] in top_k_preds[i] for i in range(len(y_true)))
    return correct / len(y_true)


def compute_topk_accuracies(y_true, y_pred_proba, k_values):
    num_classes = y_pred_proba.shape[1]
    return {f"Top{min(k, num_classes)}_Accuracy": topk_accuracy(y_true, y_pred_proba, min(k, num_classes))
            for k in k_values}


def normalize_proba_shape(y_pred_proba):
    y_pred_proba = np.asarray(y_pred_proba)
    if y_pred_proba.ndim == 1:
        y_pred_proba = np.column_stack([1 - y_pred_proba, y_pred_proba])
    elif y_pred_proba.ndim == 2 and y_pred_proba.shape[1] == 1:
        pos_probs = y_pred_proba[:, 0]
        y_pred_proba = np.column_stack([1 - pos_probs, pos_probs])
    return y_pred_proba


def compute_auroc(y_true, y_pred_proba):
    y_pred_proba = normalize_proba_shape(y_pred_proba)
    num_classes = len(np.unique(y_true))
    try:
        if num_classes == 2:
            return roc_auc_score(y_true, y_pred_proba[:, 1])
        y_true_bin = label_binarize(y_true, classes=np.unique(y_true))
        return roc_auc_score(y_true_bin, y_pred_proba, multi_class="ovr", average="weighted")
    except ValueError as e:
        print(f"[AUROC] Could not compute: {e}")
        return float("nan")


def compute_per_class_auroc(y_true, y_pred_proba):
    y_pred_proba = normalize_proba_shape(y_pred_proba)
    classes = np.unique(y_true)
    num_classes = len(classes)
    y_true_bin = label_binarize(y_true, classes=classes)
    per_class = {}
    if num_classes == 2:
        try:
            per_class[f"class_{classes[1]}_auroc"] = roc_auc_score(y_true_bin[:, 0], y_pred_proba[:, 1])
        except ValueError:
            per_class[f"class_{classes[1]}_auroc"] = float("nan")
    else:
        for i in range(num_classes):
            try:
                if len(np.unique(y_true_bin[:, i])) < 2:
                    per_class[f"class_{classes[i]}_auroc"] = float("nan")
                    continue
                per_class[f"class_{classes[i]}_auroc"] = roc_auc_score(y_true_bin[:, i], y_pred_proba[:, i])
            except ValueError:
                per_class[f"class_{classes[i]}_auroc"] = float("nan")
    return per_class


def compute_precision_recall(y_true, y_pred):
    macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    per_class_prec = precision_score(y_true, y_pred, average=None, zero_division=0)
    per_class_rec = recall_score(y_true, y_pred, average=None, zero_division=0)
    unique_classes = np.unique(y_true)
    per_class_dict = {}
    for cls, p, r in zip(unique_classes, per_class_prec, per_class_rec):
        per_class_dict[f"class_{cls}_precision"] = p
        per_class_dict[f"class_{cls}_recall"] = r
    return {"Macro_Precision": macro_precision, "Weighted_Precision": weighted_precision,
            "Macro_Recall": macro_recall, "Weighted_Recall": weighted_recall, **per_class_dict}


def compute_f1_scores(y_true, y_pred):
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    per_class = f1_score(y_true, y_pred, average=None)
    unique_classes = np.unique(y_true)
    per_class_dict = {f"class_{cls}_f1": f1 for cls, f1 in zip(unique_classes, per_class)}
    return {"macro_f1": macro_f1, "weighted_f1": weighted_f1,
            "per_class_f1": per_class_dict, "per_class_f1_values": per_class}


def compute_cohen_kappa(y_true, y_pred):
    return {
        "Cohen_Kappa": cohen_kappa_score(y_true, y_pred, weights=None),
        "Cohen_Kappa_Quadratic": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        "Cohen_Kappa_Linear_Wt": cohen_kappa_score(y_true, y_pred, weights="linear"),
    }


def compute_map(y_true, y_pred_proba):
    y_pred_proba = normalize_proba_shape(y_pred_proba)
    classes = np.unique(y_true)
    num_classes = len(classes)
    y_true_bin = label_binarize(y_true, classes=classes)
    per_class_ap, ap_values = {}, []
    if num_classes == 2:
        try:
            ap = average_precision_score(y_true_bin[:, 0], y_pred_proba[:, 1])
        except ValueError:
            ap = float("nan")
        per_class_ap[f"class_{classes[1]}_AP"] = ap
        ap_values.append(ap)
    else:
        for i in range(num_classes):
            try:
                ap = (float("nan") if len(np.unique(y_true_bin[:, i])) < 2
                      else average_precision_score(y_true_bin[:, i], y_pred_proba[:, i]))
            except ValueError:
                ap = float("nan")
            per_class_ap[f"class_{classes[i]}_AP"] = ap
            ap_values.append(ap)
    valid = [v for v in ap_values if not np.isnan(v)]
    return {"mAP": np.mean(valid) if valid else float("nan"), **per_class_ap}, np.array(ap_values)


def build_full_metrics(y_true, y_pred, y_pred_proba, k_values=(1, 5, 10)):
    f1_results = compute_f1_scores(y_true, y_pred)
    pr_results = compute_precision_recall(y_true, y_pred)
    topk_results = compute_topk_accuracies(y_true, y_pred_proba, k_values)
    auroc = compute_auroc(y_true, y_pred_proba)
    per_cls_auroc = compute_per_class_auroc(y_true, y_pred_proba)
    kappa_results = compute_cohen_kappa(y_true, y_pred)
    map_results, ap_values = compute_map(y_true, y_pred_proba)
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Acc": balanced_accuracy_score(y_true, y_pred),
        "AUROC": auroc,
        "Macro_F1": f1_results["macro_f1"],
        "Weighted_F1": f1_results["weighted_f1"],
        **f1_results["per_class_f1"], **pr_results, **topk_results,
        **per_cls_auroc, **kappa_results, **map_results,
    }
    return metrics, f1_results, ap_values

### Plot functions (same plots, `plt.show()` instead of `wandb.log`)

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names=None):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix"); plt.ylabel("True Label"); plt.xlabel("Predicted Label")
    plt.tight_layout(); plt.show()
    return cm


def plot_roc_curves(y_true, y_pred_proba, num_classes, class_names=None):
    y_true_bin = label_binarize(y_true, classes=np.arange(num_classes))
    plt.figure(figsize=(10, 8))
    if num_classes == 2:
        fpr, tpr, _ = roc_curve(y_true, y_pred_proba[:, 1])
        auc = roc_auc_score(y_true, y_pred_proba[:, 1])
        plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.3f})")
    else:
        for i in range(num_classes):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
            auc = roc_auc_score(y_true_bin[:, i], y_pred_proba[:, i])
            name = class_names[i] if class_names and i < len(class_names) else f"Class {i}"
            plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "k--", label="Random Classifier")
    plt.xlim([0, 1]); plt.ylim([0, 1.05])
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curves"); plt.legend(loc="lower right"); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


def plot_f1_scores(f1_results, class_names=None):
    per_class_f1 = f1_results["per_class_f1_values"]
    classes = list(range(len(per_class_f1)))
    labels = class_names if class_names and len(class_names) == len(classes) else [f"Class {c}" for c in classes]
    plt.figure(figsize=(12, 6))
    bars = plt.bar(classes, per_class_f1, alpha=0.8, color="steelblue")
    plt.axhline(y=f1_results["macro_f1"], color="red", linestyle="--", label=f'Macro F1: {f1_results["macro_f1"]:.4f}')
    plt.axhline(y=f1_results["weighted_f1"], color="green", linestyle="--", label=f'Weighted F1: {f1_results["weighted_f1"]:.4f}')
    for bar, f1 in zip(bars, per_class_f1):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{f1:.3f}", ha="center", va="bottom", fontsize=10)
    plt.xlabel("Class"); plt.ylabel("F1 Score"); plt.title("Per-Class F1 Scores")
    plt.xticks(classes, labels, rotation=30, ha="right")
    plt.legend(); plt.grid(True, alpha=0.3, axis="y"); plt.ylim(0, 1.05); plt.tight_layout(); plt.show()


def plot_topk_accuracy(topk_results):
    labels, values = list(topk_results.keys()), list(topk_results.values())
    plt.figure(figsize=(8, 5))
    bars = plt.bar(labels, values, color="mediumpurple", alpha=0.85)
    for bar, v in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005, f"{v:.4f}", ha="center", va="bottom", fontsize=11)
    plt.xlabel("k"); plt.ylabel("Accuracy"); plt.title("Top-k Accuracy")
    plt.ylim(0, 1.05); plt.grid(True, alpha=0.3, axis="y"); plt.tight_layout(); plt.show()


def plot_precision_recall(pr_results, class_names=None):
    classes = sorted(set(int(k.split("_")[1]) for k in pr_results if k.startswith("class_") and k.endswith("_precision")))
    precs = [pr_results[f"class_{c}_precision"] for c in classes]
    recs = [pr_results[f"class_{c}_recall"] for c in classes]
    labels = [class_names[c] if class_names and c < len(class_names) else f"Class {c}" for c in classes]
    x = np.arange(len(classes)); width = 0.35
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - width / 2, precs, width, label="Precision", alpha=0.8, color="steelblue")
    ax.bar(x + width / 2, recs, width, label="Recall", alpha=0.8, color="coral")
    ax.set_xlabel("Class"); ax.set_ylabel("Score"); ax.set_title("Per-Class Precision and Recall")
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.legend(); ax.grid(True, alpha=0.3, axis="y"); plt.tight_layout(); plt.show()


def plot_per_class_auroc(per_cls_auroc, class_names=None):
    keys = sorted(per_cls_auroc.keys())
    values = [per_cls_auroc[k] for k in keys]
    if class_names:
        labels = [class_names[int(k.split("_")[1])] if int(k.split("_")[1]) < len(class_names) else k for k in keys]
    else:
        labels = keys
    plt.figure(figsize=(12, 6))
    bars = plt.bar(labels, values, alpha=0.8, color="darkorange")
    for bar, v in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
    plt.xlabel("Class"); plt.ylabel("AUROC"); plt.title("Per-Class AUROC (OvR)")
    plt.ylim(0, 1.05); plt.xticks(rotation=30, ha="right")
    plt.grid(True, alpha=0.3, axis="y"); plt.tight_layout(); plt.show()


def plot_map(ap_values, mean_ap, class_names=None):
    classes = list(range(len(ap_values)))
    labels = class_names if class_names and len(class_names) == len(classes) else [f"Class {c}" for c in classes]
    plt.figure(figsize=(12, 6))
    bars = plt.bar(classes, ap_values, alpha=0.8, color="mediumseagreen")
    plt.axhline(y=mean_ap, color="crimson", linestyle="--", label=f"mAP: {mean_ap:.4f}")
    for bar, v in zip(bars, ap_values):
        if not np.isnan(v):
            plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
    plt.xlabel("Class"); plt.ylabel("Average Precision"); plt.title("Per-Class Average Precision (AP) and mAP")
    plt.xticks(classes, labels, rotation=30, ha="right")
    plt.legend(); plt.ylim(0, 1.05); plt.grid(True, alpha=0.3, axis="y"); plt.tight_layout(); plt.show()


def plot_kappa_summary(kappa_results):
    labels, values = list(kappa_results.keys()), list(kappa_results.values())
    plt.figure(figsize=(7, 4))
    colors = ["steelblue" if v >= 0.8 else "goldenrod" if v >= 0.6 else "tomato" for v in values]
    bars = plt.bar(labels, values, color=colors, alpha=0.85)
    for thresh, label, ls in [(0.8, "Strong (0.8)", "--"), (0.6, "Moderate (0.6)", ":")]:
        plt.axhline(y=thresh, color="gray", linestyle=ls, linewidth=1, label=label)
    for bar, v in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{v:.4f}", ha="center", va="bottom", fontsize=10)
    plt.ylabel("Kappa"); plt.title("Cohen's Kappa variants")
    plt.ylim(min(0, min(values)) - 0.05, 1.05); plt.xticks(rotation=15); plt.legend(fontsize=9)
    plt.grid(True, alpha=0.3, axis="y"); plt.tight_layout(); plt.show()


def plot_feature_importance(model, feature_names=None, top_k=20):
    if not hasattr(model, "coef_"):
        return
    coef = np.abs(model.coef_)
    importance = np.mean(coef, axis=0) if coef.shape[0] > 1 else coef[0]
    top_indices = np.argsort(importance)[-top_k:][::-1]
    top_importance = importance[top_indices]
    if feature_names is None:
        feature_names = [f"Feature {i}" for i in range(len(importance))]
    top_features = [feature_names[i] for i in top_indices]
    plt.figure(figsize=(12, 6))
    plt.barh(range(len(top_features)), top_importance)
    plt.yticks(range(len(top_features)), top_features)
    plt.xlabel("Absolute Coefficient Value"); plt.title(f"Top {top_k} Most Important Features")
    plt.tight_layout(); plt.show()


def plot_prediction_distribution(y_true, y_pred, y_pred_proba):
    max_probs = np.max(y_pred_proba, axis=1)
    correct_mask = (y_true == y_pred)
    correct_probs, incorrect_probs = max_probs[correct_mask], max_probs[~correct_mask]
    plt.figure(figsize=(10, 6))
    plt.hist(correct_probs, bins=50, alpha=0.5, label="Correct Predictions", color="green")
    plt.hist(incorrect_probs, bins=50, alpha=0.5, label="Incorrect Predictions", color="red")
    plt.xlabel("Prediction Confidence"); plt.ylabel("Count")
    plt.title("Prediction Confidence Distribution"); plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"avg correct confidence: {np.mean(correct_probs):.4f}  "
          f"avg incorrect confidence: {np.mean(incorrect_probs) if len(incorrect_probs) else float('nan'):.4f}")


def plot_class_distribution(train_labels, test_labels, class_names=None):
    unique_classes = np.unique(np.concatenate([train_labels, test_labels]))
    train_counts = [np.sum(train_labels == c) for c in unique_classes]
    test_counts = [np.sum(test_labels == c) for c in unique_classes]
    labels = [class_names[c] if class_names and c < len(class_names) else f"Class {c}" for c in unique_classes]
    x = np.arange(len(unique_classes)); width = 0.35
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - width / 2, train_counts, width, label="Train", alpha=0.8)
    ax.bar(x + width / 2, test_counts, width, label="Test", alpha=0.8)
    ax.set_xlabel("Class"); ax.set_ylabel("Count"); ax.set_title("Class Distribution")
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.legend(); ax.grid(True, alpha=0.3, axis="y"); plt.tight_layout(); plt.show()


def plot_per_class_metrics_grid(y_true, y_pred, class_names=None):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    classes = sorted([k for k in report if k not in ["accuracy", "macro avg", "weighted avg"]], key=lambda k: int(k))
    labels = [class_names[int(c)] if class_names and int(c) < len(class_names) else c for c in classes]
    df = pd.DataFrame([
        {"Class": lbl, "Precision": report[c]["precision"], "Recall": report[c]["recall"],
         "F1-Score": report[c]["f1-score"], "Support": report[c]["support"]}
        for c, lbl in zip(classes, labels)
    ])
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    for ax, col, title in zip(axes.flat, ["Precision", "Recall", "F1-Score", "Support"],
                               ["Precision per Class", "Recall per Class", "F1-Score per Class", "Support per Class"]):
        ax.bar(df["Class"], df[col]); ax.set_title(title); ax.set_ylabel(col)
        ax.tick_params(axis="x", rotation=45); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


def run_full_eval(model_label, y_test, test_pred, test_proba, class_names, k_values=(1, 5)):
    """Computes all metrics + shows every plot for one fitted model's test predictions."""
    metrics, f1_results, ap_values = build_full_metrics(y_test, test_pred, test_proba, k_values=k_values)

    print(f"\n=== {model_label} ===")
    for key in ["Accuracy", "Balanced_Acc", "AUROC", "mAP", "Macro_F1", "Weighted_F1",
                "Macro_Precision", "Macro_Recall", "Cohen_Kappa", "Cohen_Kappa_Quadratic",
                *[k for k in metrics if k.startswith("Top")]]:
        print(f"  {key}: {metrics.get(key, float('nan')):.4f}")
    print("\n" + classification_report(y_test, test_pred, target_names=class_names, zero_division=0))

    num_classes = len(np.unique(y_test))
    topk_only = {k: v for k, v in metrics.items() if k.startswith("Top") and k.endswith("_Accuracy")}
    per_cls_auroc = {k: v for k, v in metrics.items() if k.endswith("_auroc")}
    pr_only = {k: v for k, v in metrics.items() if k.startswith("class_") and (k.endswith("_precision") or k.endswith("_recall"))}
    kappa_only = {k: v for k, v in metrics.items() if k.startswith("Cohen_Kappa")}

    plot_confusion_matrix(y_test, test_pred, class_names)
    plot_roc_curves(y_test, test_proba, num_classes, class_names)
    plot_f1_scores(f1_results, class_names)
    plot_topk_accuracy(topk_only)
    plot_precision_recall(pr_only, class_names)
    plot_per_class_auroc(per_cls_auroc, class_names)
    plot_map(ap_values, metrics["mAP"], class_names)
    plot_kappa_summary(kappa_only)
    plot_prediction_distribution(y_test, test_pred, test_proba)
    plot_per_class_metrics_grid(y_test, test_pred, class_names)

    return metrics, f1_results

### Load the features just extracted

Reads directly from `output_base` (the PanNuke run's `train_data`/`val_data`/
`test_data` folders written by the extraction cells above) -- same
train+val-combined convention as `get_data()` in your script.

In [ ]:
train_features, train_labels_raw, _, h5_class_names = load_combined_features(output_base / "train_data" / "train_features.h5")

val_file = output_base / "val_data" / "val_features.h5"
if val_file.exists():
    val_features, val_labels_raw, _, _ = load_combined_features(val_file)
    X_train = np.vstack([train_features, val_features])
    y_train = np.concatenate([train_labels_raw, val_labels_raw]).flatten()
else:
    X_train = train_features
    y_train = train_labels_raw.flatten()

X_test, y_test_raw, _, test_class_names = load_combined_features(output_base / "test_data" / "test_features.h5")
y_test = y_test_raw.flatten()

if X_train.ndim > 2:
    X_train = X_train.reshape(X_train.shape[0], -1)
if X_test.ndim > 2:
    X_test = X_test.reshape(X_test.shape[0], -1)

eval_class_names = h5_class_names or test_class_names or class_names or DEFAULT_PANNUKE_CLASSES

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}  y_test:  {y_test.shape}")
print("classes:", eval_class_names)

### Logistic regression

Same `C = 1 / (100 / (M*C))` regularization and `solver="lbfgs"` as
`train_and_evaluate_logistic_regression()`.

In [ ]:
K_VALUES = (1, 5)

plot_class_distribution(y_train, y_test, eval_class_names)

M = X_train.shape[1]
C_num = len(np.unique(y_train))
l2 = 100 / (M * C_num)

logreg = LogisticRegression(C=1 / l2, max_iter=1000, solver="lbfgs")
logreg.fit(X_train, y_train)

logreg_pred = logreg.predict(X_test)
logreg_proba = logreg.predict_proba(X_test)

logreg_metrics, logreg_f1 = run_full_eval("Logistic Regression", y_test, logreg_pred, logreg_proba, eval_class_names, K_VALUES)
logreg_metrics["Final_Loss"] = log_loss(y_test, logreg_proba, labels=np.unique(y_train))
print(f"Final_Loss: {logreg_metrics['Final_Loss']:.4f}")
plot_feature_importance(logreg, top_k=min(20, M))

### kNN (n_neighbors = 1 and 20)

In [ ]:
knn_results = {}
for n_neighbors in [1, 20]:
    knn = KNeighborsClassifier(n_neighbors=n_neighbors)
    knn.fit(X_train, y_train)
    knn_pred = knn.predict(X_test)
    knn_proba = knn.predict_proba(X_test)
    knn_metrics, knn_f1 = run_full_eval(f"{n_neighbors}-NN", y_test, knn_pred, knn_proba, eval_class_names, K_VALUES)
    knn_results[n_neighbors] = knn_metrics

### UMAP of the embeddings (optional -- needs `umap-learn`)

In [ ]:
if _UMAP_AVAILABLE:
    for data, labels, title in [(X_train, y_train, "train"), (X_test, y_test, "test")]:
        reducer = umap.UMAP()
        embedding = reducer.fit_transform(data)
        plt.figure(figsize=(10, 8))
        scatter = plt.scatter(embedding[:, 0], embedding[:, 1], c=labels, s=3, cmap="Spectral", alpha=0.7)
        cbar = plt.colorbar(scatter, ticks=range(len(eval_class_names)))
        cbar.ax.set_yticklabels(eval_class_names)
        plt.title(f"UMAP -- {title}"); plt.xlabel("UMAP 1"); plt.ylabel("UMAP 2")
        plt.tight_layout(); plt.show()
else:
    print("Skipping UMAP -- umap-learn not installed.")

## Next

This covers one checkpoint on PanNuke end to end: load -> extract -> evaluate
-> visualize. To extend it:
- Swap `MODEL_NAME`/`PATCH_SIZE`/`CHECKPOINT_PATH` and rerun to compare
  another checkpoint (rerun from the Config cell down).
- Swap `setup_pannuke_datasets` for another dataset in `feature_extraction.py`
  (conic, immunocto, nucls, ...) to move to a different dataset.
- For comparing many runs at once with mean+/-std over pooled k-fold CV,
  that is what `run_ablation_sweep.py` + the `--table` mode of your
  evaluation script are for -- this notebook intentionally does not
  reimplement that batch path.